# DFI Video Maker — batch renderer (Sheet + Drive)

Renders a spinning-record video for every `Render? = TRUE` row in the DFI track
sheet: cover art spinning like a record (with motion blur), the DFI branding
overlaid, and a snippet of the track underneath. Finished MP4s are saved to a
Google Drive folder.

**How to run:** check the **Config** cell below, then **Runtime → Run all** and
sign in with the shared Google account when prompted. Everything else is automatic.


## 1 · Config  ← check these

In [ ]:
# ---- IDs (already filled in for the DFI account) ---------------------------
SHEET_ID               = "1S_MFhIt0V8OJWZ8IMYAFf9_bQJ2WVMcJpYLe5uCqlRY"
WORKSHEET_NAME         = None        # None = first tab; or a tab name (str) / index (int)
DRIVE_OUTPUT_FOLDER_ID = "1iW4LFcTWxxA2qze4jga0O7s6EGG5c8cw"   # finished videos land here

# ---- Sheet column headers (edit here if you rename a column in the sheet) ---
# These must match the header text in your sheet EXACTLY (spelling, spaces, case).
COL_TRACK      = "Track"
COL_ARTIST     = "Artist"
COL_AUDIO      = "Drive audio file"
COL_ARTWORK    = "Drive artwork file*"
COL_CLIP_START = "Clip start"
COL_RENDER     = "Render?"

# ---- Branding overlay (optional) -------------------------------------------
# A transparent PNG the SAME size as the canvas, kept in Google Drive.
# Paste its share link below. To change the branding, paste a different link.
# Leave it as "" for no overlay.
OVERLAY_DRIVE_LINK     = "https://drive.google.com/file/d/1kj_GVgd9Fy1hMR8GLMTswzqDOZEHdp_Q/view"  # paste a different link to change branding

# ---- Look & feel -----------------------------------------------------------
CLIP_LENGTH_SECONDS = 25             # clip length for every row
SPIN_PERIOD_SECONDS = 6              # seconds per full rotation
FPS                 = 30
CANVAS_W            = 1080           # 1080 x 1080 = 1:1 square
CANVAS_H            = 1080           # set 1350 for 4:5 (and use a 1080x1350 overlay)
CIRCLE_DIAMETER     = 790            # diameter of the spinning record
BG_COLOUR           = "black"
MOTION_BLUR_SAMPLES = 10             # 1 = no blur
SHUTTER_FRACTION    = 0.7            # blur amount (0.5 = 180-degree shutter)

## 2 · Install dependencies
(ffmpeg via apt; the Python libs are already in Colab but we pin them to be safe.)

In [ ]:
import subprocess, sys
print("Installing ffmpeg ...")
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "pillow", "mutagen", "numpy", "gspread",
                "google-api-python-client", "google-auth-httplib2"], check=True)
print("Dependencies ready.")

## 3 · Sign in to Google
Authorises **you** for Sheets + Drive. Use the shared DFI account when the pop-up asks.

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
from googleapiclient.discovery import build

creds, _ = default()
gc = gspread.authorize(creds)
drive = build("drive", "v3")
print("Signed in — connected to Google Sheets + Drive.")

## 4 · Load the render engine
Downloads the latest engine from GitHub and imports it, so you always run the newest version — no need to re-upload the notebook when the engine improves.

In [ ]:
# Fetch the latest engine from GitHub (this is what makes the notebook a thin
# "launcher": the engine lives in the repo, so improvements arrive automatically).
import urllib.request
ENGINE_URL = "https://raw.githubusercontent.com/tiredlabrador/dfi-video-maker/main/generate_video.py"
urllib.request.urlretrieve(ENGINE_URL, "generate_video.py")
print("Engine downloaded from GitHub.")

In [ ]:
import importlib, generate_video
importlib.reload(generate_video)
from generate_video import (RenderConfig, render_video,
                            NoArtworkError, RenderError, sanitise_filename)

CFG = RenderConfig(
    clip_length_seconds=CLIP_LENGTH_SECONDS,
    spin_period_seconds=SPIN_PERIOD_SECONDS,
    fps=FPS, canvas_w=CANVAS_W, canvas_h=CANVAS_H,
    circle_diameter=CIRCLE_DIAMETER, bg_colour=BG_COLOUR,
    motion_blur_samples=MOTION_BLUR_SAMPLES, shutter_fraction=SHUTTER_FRACTION,
)
print("Engine loaded.")

## 5 · Drive / URL helpers
Resolve a Drive share link (or a plain image URL) to a downloaded file, and upload results back to Drive.

In [ ]:
import io, os, re, tempfile, requests
from urllib.parse import urlparse
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

def extract_drive_id(link):
    """Pull a Drive file id out of the common share-link shapes (or a bare id)."""
    link = str(link).strip()
    for pat in (r"/d/([A-Za-z0-9_-]{20,})", r"[?&]id=([A-Za-z0-9_-]{20,})"):
        m = re.search(pat, link)
        if m:
            return m.group(1)
    if re.fullmatch(r"[A-Za-z0-9_-]{20,}", link):
        return link
    return None

def _drive_name(file_id):
    meta = drive.files().get(fileId=file_id, fields="name",
                             supportsAllDrives=True).execute()
    return meta.get("name", "")

def _download_drive(file_id, dest):
    req = drive.files().get_media(fileId=file_id, supportsAllDrives=True)
    with io.FileIO(dest, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, req)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return dest

def fetch_source(link, dest_base):
    """
    Download `link` to `dest_base` + an inferred extension; return the path.
    Handles Drive share links, bare Drive ids, and plain http(s) image URLs.
    """
    link = str(link).strip()
    is_drive = ("drive.google.com" in link) or (not link.lower().startswith("http"))
    if is_drive:
        file_id = extract_drive_id(link)
        if not file_id:
            raise ValueError(f"Could not parse a Drive id from: {link!r}")
        ext = os.path.splitext(_drive_name(file_id))[1]
        return _download_drive(file_id, dest_base + ext)
    ext = os.path.splitext(urlparse(link).path)[1]
    dest = dest_base + ext
    r = requests.get(link, timeout=60)
    r.raise_for_status()
    with open(dest, "wb") as fh:
        fh.write(r.content)
    return dest

def upload_to_drive(local_path, name, folder_id):
    meta = {"name": name, "parents": [folder_id]}
    media = MediaFileUpload(local_path, mimetype="video/mp4", resumable=True)
    return drive.files().create(body=meta, media_body=media,
                                fields="id,webViewLink",
                                supportsAllDrives=True).execute()

def get_or_create_folder(name, parent_id):
    """Return the id of a subfolder `name` inside `parent_id`, creating it if
    it doesn't exist. Re-running a batch reuses the same folder (no duplicates)."""
    safe = name.replace("\\", "\\\\").replace("'", "\\'")
    q = ("name = '" + safe + "' and mimeType = 'application/vnd.google-apps.folder' "
         "and '" + parent_id + "' in parents and trashed = false")
    hits = drive.files().list(q=q, spaces="drive", fields="files(id, name)",
                              supportsAllDrives=True,
                              includeItemsFromAllDrives=True).execute().get("files", [])
    if hits:
        return hits[0]["id"]
    meta = {"name": name, "mimeType": "application/vnd.google-apps.folder",
            "parents": [parent_id]}
    return drive.files().create(body=meta, fields="id",
                                supportsAllDrives=True).execute()["id"]

print("Helpers ready.")

## 6 · Read the track sheet

In [ ]:
sh = gc.open_by_key(SHEET_ID)
if WORKSHEET_NAME in (None, ""):
    ws = sh.sheet1
elif isinstance(WORKSHEET_NAME, int):
    ws = sh.get_worksheet(WORKSHEET_NAME)
else:
    ws = sh.worksheet(WORKSHEET_NAME)

rows = ws.get_all_records()   # list of dicts keyed by the header row
print(f"Read {len(rows)} data rows from tab '{ws.title}'.")

## 7 · Batch render
Downloads the branding overlay once, then renders every `Render? = TRUE` row. One bad row never halts the batch — each is caught, logged, and the run continues. A summary prints at the end.

In [ ]:
# --- branding overlay: download once from Drive (optional) ------------------
CFG.overlay_path = None
if OVERLAY_DRIVE_LINK.strip():
    try:
        _ov_dir = tempfile.mkdtemp()
        CFG.overlay_path = fetch_source(OVERLAY_DRIVE_LINK,
                                        os.path.join(_ov_dir, "overlay"))
        print("Branding overlay loaded from Drive.")
    except Exception as exc:
        print(f"WARNING: could not load overlay ({exc}). Rendering without branding.")
        CFG.overlay_path = None

def is_true(v):
    return v is True or str(v).strip().upper() in ("TRUE", "YES", "1")

rendered, skipped, failed = [], [], []
batch_folder_id = None   # a subfolder named after the tab, created on first render

with tempfile.TemporaryDirectory() as work:
    for i, row in enumerate(rows, start=2):   # row 1 is the header
        track  = str(row.get(COL_TRACK, "")).strip()
        artist = str(row.get(COL_ARTIST, "")).strip()
        label  = f"row {i}: {artist} - {track}".strip(" -")

        if not is_true(row.get(COL_RENDER)):
            continue

        audio_link = str(row.get(COL_AUDIO, "")).strip()
        if not audio_link:
            skipped.append((label, "Audio file blank"))
            print(f"SKIP  {label} — Audio file blank")
            continue

        try:
            audio_path = fetch_source(audio_link, os.path.join(work, f"audio_{i}"))

            artwork_link = str(row.get(COL_ARTWORK, "")).strip()
            artwork_path = None
            if artwork_link:
                artwork_path = fetch_source(artwork_link, os.path.join(work, f"art_{i}"))

            name = sanitise_filename(f"{artist} - {track}") + ".mp4"
            out_path = os.path.join(work, name)
            render_video(audio_path, artwork_path,
                         row.get(COL_CLIP_START, "0:00"), out_path, CFG)

            if batch_folder_id is None:      # create the batch subfolder on first success
                batch_folder_id = get_or_create_folder(ws.title, DRIVE_OUTPUT_FOLDER_ID)
                print(f"Saving videos to output subfolder: {ws.title!r}")
            info = upload_to_drive(out_path, name, batch_folder_id)
            rendered.append((label, name, info.get("webViewLink", "")))
            print(f"OK    {label} -> {name}")

        except NoArtworkError:
            skipped.append((label, "no artwork (no override and no embedded art)"))
            print(f"SKIP  {label} — no artwork")
        except Exception as exc:
            failed.append((label, str(exc).splitlines()[0] if str(exc) else repr(exc)))
            print(f"FAIL  {label} — {exc}")

print("\n" + "=" * 64)
print("  BATCH SUMMARY")
print("=" * 64)
print(f"  Rendered OK : {len(rendered)}")
for label, name, link in rendered:
    print(f"      - {name}   {link}")
print(f"  Skipped     : {len(skipped)}")
for label, reason in skipped:
    print(f"      - {label}  ({reason})")
print(f"  Failed      : {len(failed)}")
for label, reason in failed:
    print(f"      - {label}  ({reason})")
print("=" * 64)